In [1]:
!pip install streamlit pyjwt bcrypt pyngrok textstat plotly PyPDF2 streamlit-option-menu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 41.9 MB/s eta 0:00:00


In [2]:
%%writefile database.py
import sqlite3
import bcrypt
import json
from datetime import datetime, timedelta

DB_NAME = 'policynav_m2.db'

def get_conn():
    return sqlite3.connect(DB_NAME)

def init_database():
    conn = get_conn()
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        password TEXT NOT NULL,
        security_question TEXT NOT NULL,
        security_answer TEXT NOT NULL,
        failed_attempts INTEGER DEFAULT 0,
        locked_until TEXT DEFAULT NULL,
        password_history TEXT DEFAULT '[]',
        is_verified INTEGER DEFAULT 0,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS otp_store (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        email TEXT NOT NULL,
        otp TEXT NOT NULL,
        purpose TEXT NOT NULL,
        expires_at TEXT NOT NULL,
        used INTEGER DEFAULT 0
    )''')
    conn.commit()
    conn.close()
    print("Database initialized!")

def hash_password(password):
    return bcrypt.hashpw(password.encode(), bcrypt.gensalt()).decode()

def verify_password(password, hashed):
    return bcrypt.checkpw(password.encode(), hashed.encode())

def create_user(username, email, password, security_question, security_answer):
    try:
        conn = get_conn()
        c = conn.cursor()
        hashed_pw = hash_password(password)
        hashed_answer = hash_password(security_answer.lower())
        history = json.dumps([hashed_pw])
        c.execute(
            'INSERT INTO users (username,email,password,security_question,security_answer,password_history) VALUES (?,?,?,?,?,?)',
            (username, email, hashed_pw, security_question, hashed_answer, history)
        )
        conn.commit()
        conn.close()
        return True, "User registered! Check your email for OTP."
    except sqlite3.IntegrityError:
        return False, "Email already exists!"
    except Exception as e:
        return False, str(e)

def get_user_by_email(email):
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT * FROM users WHERE email=?', (email,))
    user = c.fetchone()
    conn.close()
    return user

def mark_verified(email):
    conn = get_conn()
    c = conn.cursor()
    c.execute('UPDATE users SET is_verified=1 WHERE email=?', (email,))
    conn.commit()
    conn.close()

def record_failed_attempt(email):
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT failed_attempts FROM users WHERE email=?', (email,))
    row = c.fetchone()
    if row:
        new_count = row[0] + 1
        locked_until = None
        if new_count >= 3:
            locked_until = (datetime.utcnow() + timedelta(minutes=5)).isoformat()
        c.execute('UPDATE users SET failed_attempts=?, locked_until=? WHERE email=?',
                  (new_count, locked_until, email))
    conn.commit()
    conn.close()

def reset_failed_attempts(email):
    conn = get_conn()
    c = conn.cursor()
    c.execute('UPDATE users SET failed_attempts=0, locked_until=NULL WHERE email=?', (email,))
    conn.commit()
    conn.close()

def is_account_locked(email):
    user = get_user_by_email(email)
    if user and user[7]:
        locked_until = datetime.fromisoformat(user[7])
        if datetime.utcnow() < locked_until:
            remaining = int((locked_until - datetime.utcnow()).total_seconds())
            return True, remaining
        else:
            reset_failed_attempts(email)
    return False, 0

def is_password_reused(email, new_password):
    user = get_user_by_email(email)
    if user:
        history = json.loads(user[8])
        for old_hash in history:
            if verify_password(new_password, old_hash):
                return True
    return False

def update_password(email, new_password):
    try:
        conn = get_conn()
        c = conn.cursor()
        hashed = hash_password(new_password)
        c.execute('SELECT password_history FROM users WHERE email=?', (email,))
        row = c.fetchone()
        history = json.loads(row[0]) if row else []
        history.append(hashed)
        if len(history) > 5:
            history = history[-5:]
        c.execute('UPDATE users SET password=?, password_history=? WHERE email=?',
                  (hashed, json.dumps(history), email))
        conn.commit()
        conn.close()
        return True, "Password updated successfully!"
    except Exception as e:
        return False, str(e)

def store_otp(email, otp, purpose):
    conn = get_conn()
    c = conn.cursor()
    expires = (datetime.utcnow() + timedelta(minutes=10)).isoformat()
    c.execute('DELETE FROM otp_store WHERE email=? AND purpose=?', (email, purpose))
    c.execute('INSERT INTO otp_store (email,otp,purpose,expires_at) VALUES (?,?,?,?)',
              (email, otp, purpose, expires))
    conn.commit()
    conn.close()

def verify_otp(email, otp, purpose):
    conn = get_conn()
    c = conn.cursor()
    c.execute('SELECT otp, expires_at, used FROM otp_store WHERE email=? AND purpose=? ORDER BY id DESC LIMIT 1',
              (email, purpose))
    row = c.fetchone()
    conn.close()
    if not row:
        return False, "OTP not found."
    stored_otp, expires_at, used = row
    if used:
        return False, "OTP already used."
    if datetime.utcnow() > datetime.fromisoformat(expires_at):
        return False, "OTP has expired."
    if otp.strip() != stored_otp:
        return False, "Invalid OTP."
    conn2 = get_conn()
    c2 = conn2.cursor()
    c2.execute('UPDATE otp_store SET used=1 WHERE email=? AND purpose=?', (email, purpose))
    conn2.commit()
    conn2.close()
    return True, "OTP verified!"

init_database()

def get_all_users():
    conn = get_conn()
    c = conn.cursor()
    c.execute("SELECT id, username, email, failed_attempts, is_verified, locked_until, created_at FROM users ORDER BY created_at DESC")
    users = c.fetchall()
    conn.close()
    return users

def delete_user(email):
    conn = get_conn()
    c = conn.cursor()
    c.execute("DELETE FROM otp_store WHERE email=?", (email,))
    c.execute("DELETE FROM users WHERE email=?", (email,))
    conn.commit()
    conn.close()


Writing database.py


In [3]:
%%writefile jwt_auth.py
import jwt
from datetime import datetime, timedelta

def get_secret():
    try:
        from google.colab import userdata
        return userdata.get('JWT_SECRET_KEY')
    except:
        return 'policynav_fallback_secret_milestone2'

def generate_token(email, username, role='user'):
    payload = {
        'email': email,
        'username': username,
        'role': role,
        'exp': datetime.utcnow() + timedelta(hours=24),
        'iat': datetime.utcnow()
    }
    return jwt.encode(payload, get_secret(), algorithm='HS256')

def verify_token(token):
    try:
        payload = jwt.decode(token, get_secret(), algorithms=['HS256'])
        return True, payload
    except jwt.ExpiredSignatureError:
        return False, "Token expired"
    except jwt.InvalidTokenError:
        return False, "Invalid token"


Writing jwt_auth.py


In [4]:
%%writefile otp_service.py
import random
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

def get_email_config():
    try:
        from google.colab import userdata
        return userdata.get('EMAIL_ID'), userdata.get('EMAIL_APP_PASSWORD')
    except:
        return None, None

def generate_otp():
    return str(random.randint(100000, 999999))

def send_otp_email(recipient_email, otp, purpose='verification'):
    sender_email, app_password = get_email_config()
    if not sender_email or not app_password:
        print(f"[Email not configured] OTP for {recipient_email}: {otp}")
        return True, otp

    subjects = {
        'verification': 'PolicyNav - Email Verification OTP',
        'reset': 'PolicyNav - Password Reset OTP',
        'login': 'PolicyNav - Login OTP'
    }
    subject = subjects.get(purpose, 'PolicyNav OTP')

    body = f"""
<html><body>
<div style="font-family:Arial,sans-serif;max-width:480px;margin:auto;
            border:1px solid #ddd;border-radius:12px;overflow:hidden;">
  <div style="background:linear-gradient(135deg,#1565C0,#311B92);
              padding:24px;text-align:center;">
    <h2 style="color:white;margin:0;font-size:1.6rem;">PolicyNav</h2>
    <p style="color:rgba(255,255,255,0.85);margin:6px 0 0;">Secure Authentication</p>
  </div>
  <div style="padding:32px;">
    <p style="font-size:1rem;color:#333;">Your One-Time Password:</p>
    <div style="background:#f0f4ff;border:2px solid #1565C0;border-radius:10px;
                padding:20px;text-align:center;font-size:2.5rem;
                font-weight:700;letter-spacing:10px;color:#1565C0;
                margin:16px 0;">{otp}</div>
    <p style="color:#555;">Valid for <strong>10 minutes</strong>. Do not share this OTP.</p>
    <p style="color:#999;font-size:0.85rem;">If you did not request this, ignore this email.</p>
  </div>
</div>
</body></html>
"""

    try:
        msg = MIMEMultipart('alternative')
        msg['Subject'] = subject
        msg['From'] = sender_email
        msg['To'] = recipient_email
        msg.attach(MIMEText(body, 'html'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp:
            smtp.login(sender_email, app_password)
            smtp.sendmail(sender_email, recipient_email, msg.as_string())
        print(f"OTP email sent to {recipient_email}")
        return True, "OTP sent!"
    except Exception as e:
        print(f"Email error: {e} | OTP for testing: {otp}")
        return True, otp


Writing otp_service.py


In [5]:
%%writefile validators.py
import re

def validate_email(email):
    return bool(re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email))

def validate_password(password):
    return bool(
        re.search(r'[a-zA-Z]', password) and
        re.search(r'[0-9]', password) and
        len(password) >= 6
    )

def validate_signup_form(username, email, password, confirm, security_question, security_answer):
    errors = []
    if not username or not username.strip():
        errors.append("Username is required")
    if not email or not validate_email(email):
        errors.append("Valid email address is required")
    if not password:
        errors.append("Password is required")
    elif not validate_password(password):
        errors.append("Password must contain letters and numbers and be at least 6 characters")
    if password != confirm:
        errors.append("Passwords do not match")
    if not security_question or security_question == "Select a question":
        errors.append("Please select a security question")
    if not security_answer or not security_answer.strip():
        errors.append("Security answer is required")
    return errors


Writing validators.py


In [6]:
%%writefile readability.py
import textstat

class ReadabilityAnalyzer:
    """Readability analyzer matching the sample dashboard design."""

    def __init__(self, text):
        self.text          = text
        self.num_sentences = textstat.sentence_count(text)
        self.num_words     = textstat.lexicon_count(text, removepunct=True)
        self.num_syllables = textstat.syllable_count(text)
        self.complex_words = textstat.difficult_words(text)
        self.char_count    = textstat.char_count(text)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease":  textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "SMOG Index":           textstat.smog_index(self.text),
            "Gunning Fog":          textstat.gunning_fog(self.text),
            "Coleman-Liau":         textstat.coleman_liau_index(self.text),
        }

    def get_suggestions(self):
        score = self.get_all_metrics()
        suggestions = []
        fre   = score["Flesch Reading Ease"]
        grade = score["Flesch-Kincaid Grade"]
        fog   = score["Gunning Fog"]
        avg_sent = round(self.num_words / max(self.num_sentences, 1), 1)

        if fre < 50:
            suggestions.append("📝 Use simpler, everyday words to improve readability (Flesch score is low).")
        if avg_sent > 20:
            suggestions.append(f"✂️ Average sentence length is {avg_sent} words – aim for under 20 words per sentence.")
        if fog > 12:
            suggestions.append("🔍 Reduce polysyllabic words (3+ syllables) to lower the Gunning Fog score.")
        if grade > 12:
            suggestions.append("📚 Text reads at college level – consider simplifying for a broader public audience.")
        if self.complex_words > self.num_words * 0.15:
            pct = round(self.complex_words / self.num_words * 100)
            suggestions.append(f"💬 {pct}% of words are complex – try replacing jargon with plain-language alternatives.")
        if not suggestions:
            suggestions.append("✅ Great readability! This text is accessible to most general readers.")
        return suggestions


Writing readability.py


In [7]:
%%writefile app.py
import streamlit as st
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import time
import os
import re
import plotly.graph_objects as go
import PyPDF2

from streamlit_option_menu import option_menu

from database import (
    create_user, get_user_by_email, verify_password, update_password,
    mark_verified, record_failed_attempt, reset_failed_attempts,
    is_account_locked, is_password_reused, store_otp, verify_otp,
    get_all_users, delete_user
)
from jwt_auth import generate_token, verify_token
from otp_service import generate_otp, send_otp_email
from validators import validate_email, validate_password, validate_signup_form
from readability import ReadabilityAnalyzer

# ─── Page config ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="PolicyNav – Secure Auth",
    page_icon="⚡",
    layout="wide"
)

# ─── Neon CSS Theme ──────────────────────────────────────────────────────────
def apply_neon_theme():
    st.markdown("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Inter:wght@400;600;700&display=swap');

        .stApp {
            background-color: #0e1117;
            color: #ffffff;
        }
        h1, h2, h3 {
            color: #00ffcc !important;
            font-family: 'Share Tech Mono', monospace;
            text-shadow: 0 0 10px #00ffcc55;
        }
        h4, h5, h6 { color: #aaaaaa !important; }

        /* Inputs */
        .stTextInput > div > div > input,
        .stTextArea > div > div > textarea {
            background-color: #1f2937;
            color: #00ffcc;
            border: 1px solid #374151;
            border-radius: 6px;
            font-family: 'Share Tech Mono', monospace;
        }
        .stTextInput > div > div > input:focus,
        .stTextArea > div > div > textarea:focus {
            border-color: #00ffcc;
            box-shadow: 0 0 8px #00ffcc55;
        }

        /* Buttons */
        .stButton > button {
            background-color: #1f2937;
            color: #00ffcc;
            border: 1px solid #00ffcc;
            border-radius: 6px;
            font-family: 'Share Tech Mono', monospace;
            transition: all 0.25s ease;
            width: 100%;
        }
        .stButton > button:hover {
            background-color: #00ffcc;
            color: #0e1117;
            box-shadow: 0 0 18px #00ffcc88;
        }
        div.stButton > button[kind="primary"] {
            background-color: #00ffcc;
            color: #0e1117;
            font-weight: 700;
        }
        div.stButton > button[kind="primary"]:hover {
            box-shadow: 0 0 22px #00ffccaa;
        }

        /* Sidebar */
        section[data-testid="stSidebar"] {
            background-color: #111827;
            border-right: 1px solid #1f2937;
        }

        /* Metrics */
        [data-testid="stMetricValue"] { color: #00ffcc; font-family: 'Share Tech Mono', monospace; }
        [data-testid="stMetricLabel"] { color: #9ca3af; }

        /* Tabs */
        .stTabs [data-baseweb="tab-list"] { gap: 8px; }
        .stTabs [data-baseweb="tab"] {
            background-color: #1f2937;
            color: #9ca3af;
            border-radius: 6px;
            padding: 8px 18px;
        }
        .stTabs [aria-selected="true"] {
            background-color: #00ffcc !important;
            color: #0e1117 !important;
            font-weight: 700;
        }

        /* Expander */
        .streamlit-expanderHeader {
            background-color: #1f2937;
            color: #00ffcc;
            border-radius: 6px;
        }

        /* Progress bar */
        .stProgress > div > div { background-color: #00ffcc; }

        /* Selectbox */
        .stSelectbox > div > div {
            background-color: #1f2937;
            color: #00ffcc;
            border: 1px solid #374151;
        }

        /* File uploader */
        .stFileUploader {
            background-color: #1f2937;
            border: 1px dashed #374151;
            border-radius: 8px;
        }

        /* Strength classes */
        .s-weak   { color: #ff4b4b; font-weight: 700; font-family: monospace; }
        .s-medium { color: #ffa500; font-weight: 700; font-family: monospace; }
        .s-strong { color: #00ffcc; font-weight: 700; font-family: monospace; }

        /* Card component */
        .neon-card {
            background-color: #1f2937;
            border: 1px solid #374151;
            border-radius: 12px;
            padding: 24px;
            margin: 12px 0;
        }
        .neon-card-accent {
            background-color: #1f2937;
            border-radius: 12px;
            padding: 24px;
            margin: 12px 0;
        }

        /* Overall level badge */
        .level-badge {
            text-align: center;
            padding: 20px;
            border-radius: 10px;
            margin: 16px 0;
        }

        /* Data tables */
        .stDataFrame { background-color: #1f2937; }
        .stDataFrame td, .stDataFrame th {
            color: #ffffff !important;
            background-color: #1f2937 !important;
        }
    </style>
    """, unsafe_allow_html=True)

apply_neon_theme()

# ─── Admin credentials ───────────────────────────────────────────────────────
def get_admin_creds():
    try:
        from google.colab import userdata
        return userdata.get('ADMIN_EMAIL_ID'), userdata.get('ADMIN_PASSWORD')
    except:
        return 'admin@policynav.com', 'Admin@123'

# ─── Session state defaults ──────────────────────────────────────────────────
DEFAULTS = {
    'logged_in': False, 'token': None, 'username': None,
    'email': None, 'role': 'user',
    'page': 'login',
    'pending_email': None, 'pending_username': None,
}
for k, v in DEFAULTS.items():
    if k not in st.session_state:
        st.session_state[k] = v

SECURITY_QUESTIONS = [
    "Select a question",
    "What is your pet's name?",
    "What is your mother's maiden name?",
    "What is your favorite teacher's name?",
    "What city were you born in?",
    "What was your first car?"
]

# ─── Helpers ─────────────────────────────────────────────────────────────────
def check_password_strength(password):
    has_upper   = bool(re.search(r"[A-Z]", password))
    has_lower   = bool(re.search(r"[a-z]", password))
    has_digit   = bool(re.search(r"\d", password))
    has_special = bool(re.search(r"[!@#$%^&*(),.?\":{}|<>]", password))
    has_space   = bool(re.search(r"\s", password))
    if has_space:
        return "Weak", ["No spaces allowed"]
    is_alphanum = (has_upper or has_lower) and has_digit
    if len(password) >= 8 and is_alphanum and has_special:
        return "Strong", []
    if len(password) >= 6 and is_alphanum:
        return "Medium", ["Add special character for Strong"]
    if len(password) >= 1:
        return "Weak", ["Too short – aim for 8+ chars with letters, numbers & symbol"]
    return "Weak", ["Enter password"]

def show_strength(password):
    s, f = check_password_strength(password)
    if s == "Weak":
        st.markdown(f"Strength: <span class='s-weak'>Weak</span>", unsafe_allow_html=True)
    elif s == "Medium":
        st.markdown(f"Strength: <span class='s-medium'>Medium</span>", unsafe_allow_html=True)
    else:
        st.markdown(f"Strength: <span class='s-strong'>Strong ✓</span>", unsafe_allow_html=True)
    if f:
        st.caption(f"Hint: {', '.join(f)}")

def switch_page(page):
    st.session_state.page = page
    st.rerun()

def logout():
    for k, v in DEFAULTS.items():
        st.session_state[k] = v
    st.rerun()

# ─── Plotly gauge helper ─────────────────────────────────────────────────────
def create_gauge(value, title, min_val=0, max_val=100, color="#00ffcc"):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={'text': title, 'font': {'color': color, 'size': 13, 'family': 'Share Tech Mono'}},
        number={'font': {'color': color, 'size': 22, 'family': 'Share Tech Mono'},
                'valueformat': '.1f'},
        gauge={
            'axis': {'range': [min_val, max_val], 'tickwidth': 1,
                     'tickcolor': '#374151', 'tickfont': {'color': '#9ca3af'}},
            'bar': {'color': color},
            'bgcolor': "#1f2937",
            'borderwidth': 2,
            'bordercolor': "#374151",
            'steps': [{'range': [min_val, max_val], 'color': "#0e1117"}],
        }
    ))
    fig.update_layout(
        paper_bgcolor="#0e1117",
        font={'color': "#ffffff", 'family': "Share Tech Mono"},
        height=230,
        margin=dict(l=10, r=10, t=50, b=10)
    )
    return fig


# ═══════════════════════════════════════════════════════════════ LOGIN PAGE ══
def login_page():
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("""
        <div style="text-align:center; padding: 40px 0 20px;">
            <h1 style="font-size:2.8rem;">⚡ PolicyNav</h1>
            <p style="color:#9ca3af; font-family:'Share Tech Mono',monospace;">
            Secure Authentication System</p>
        </div>""", unsafe_allow_html=True)

        st.markdown('<div class="neon-card">', unsafe_allow_html=True)
        st.markdown("### 🔐 Login")

        with st.form("login_form"):
            email    = st.text_input("Email *", placeholder="you@example.com")
            password = st.text_input("Password *", type="password")
            submit   = st.form_submit_button("Login", type="primary")

        if submit:
            if not email or not password:
                st.error("Please fill in all fields.")
            else:
                locked, remaining = is_account_locked(email)
                if locked:
                    m, s = remaining // 60, remaining % 60
                    st.error(f"⛔ Account locked after 3 failed attempts. Try again in {m}m {s}s.")
                else:
                    user = get_user_by_email(email)
                    if user is None:
                        st.error("Email not found. Please sign up first.")
                    elif not user[9]:
                        st.warning("Email not verified. Check your inbox for the OTP.")
                    elif verify_password(password, user[3]):
                        reset_failed_attempts(email)
                        otp = generate_otp()
                        store_otp(email, otp, 'login')
                        send_otp_email(email, otp, 'login')
                        st.session_state.pending_email    = email
                        st.session_state.pending_username = user[1]
                        switch_page('verify_login_otp')
                    else:
                        record_failed_attempt(email)
                        u2   = get_user_by_email(email)
                        left = 3 - (u2[6] if u2 else 3)
                        if left > 0:
                            st.error(f"Incorrect password. {left} attempt(s) left before lockout.")
                        else:
                            st.error("Account locked for 5 minutes.")

        st.markdown("---")
        c1, c2 = st.columns(2)
        with c1:
            if st.button("Create Account"): switch_page('signup')
        with c2:
            if st.button("Forgot Password?"): switch_page('forgot_password')
        if st.button("Admin Login"): switch_page('admin_login')
        st.markdown('</div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════════ VERIFY LOGIN OTP ══
def verify_login_otp_page():
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("<h1>📧 OTP Verification</h1>", unsafe_allow_html=True)
        st.markdown('<div class="neon-card">', unsafe_allow_html=True)
        st.info(f"OTP sent to **{st.session_state.pending_email}**")
        otp_in = st.text_input("6-digit OTP", max_chars=6, placeholder="123456")
        c1, c2 = st.columns(2)
        with c1:
            if st.button("Verify & Login", type="primary"):
                ok, msg = verify_otp(st.session_state.pending_email, otp_in, 'login')
                if ok:
                    email = st.session_state.pending_email
                    uname = st.session_state.pending_username
                    token = generate_token(email, uname, 'user')
                    st.session_state.update({
                        'logged_in': True, 'token': token,
                        'username': uname, 'email': email,
                        'role': 'user'
                    })
                    st.success("✅ Login successful!")
                    time.sleep(0.8)
                    st.rerun()
                else:
                    st.error(msg)
        with c2:
            if st.button("Back"): switch_page('login')
        st.markdown('</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════════ SIGNUP PAGE ══
def signup_page():
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("<h1>⚡ PolicyNav</h1>", unsafe_allow_html=True)
        st.markdown('<div class="neon-card">', unsafe_allow_html=True)
        st.markdown("### 📝 Create Account")

        email    = st.text_input("Email Address *", placeholder="you@example.com")
        password = st.text_input("Password *", type="password",
                                  placeholder="Min 8 chars, letters + numbers + symbol")
        if password:
            show_strength(password)
        confirm  = st.text_input("Confirm Password *", type="password")
        sec_q    = st.selectbox("Security Question *", SECURITY_QUESTIONS)
        sec_a    = st.text_input("Security Answer *")
        st.caption("* All fields are mandatory")

        if st.button("Register", type="primary"):
            errors = validate_signup_form(email, email, password, confirm, sec_q, sec_a)
            # override username with email prefix for simplicity
            username = email.split("@")[0] if email else ""
            errors = validate_signup_form(username, email, password, confirm, sec_q, sec_a)
            if errors:
                for e in errors: st.error(e)
            else:
                s, _ = check_password_strength(password)
                if s == "Weak":
                    st.error("Password too weak – needs letters, numbers and a symbol.")
                else:
                    ok, msg = create_user(username, email, password, sec_q, sec_a)
                    if ok:
                        otp = generate_otp()
                        store_otp(email, otp, 'verification')
                        send_otp_email(email, otp, 'verification')
                        st.session_state.pending_email = email
                        st.success("Account created! Check email for OTP.")
                        time.sleep(1)
                        switch_page('verify_signup_otp')
                    else:
                        st.error(msg)

        st.markdown("---")
        if st.button("Back to Login"): switch_page('login')
        st.markdown('</div>', unsafe_allow_html=True)


# ══════════════════════════════════════════════════ VERIFY SIGNUP OTP ══
def verify_signup_otp_page():
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("<h1>✉️ Verify Email</h1>", unsafe_allow_html=True)
        st.markdown('<div class="neon-card">', unsafe_allow_html=True)
        st.info(f"OTP sent to **{st.session_state.pending_email}**")
        otp_in = st.text_input("6-digit OTP", max_chars=6, placeholder="123456", key="vso")
        c1, c2 = st.columns(2)
        with c1:
            if st.button("Verify Email", type="primary"):
                ok, msg = verify_otp(st.session_state.pending_email, otp_in, 'verification')
                if ok:
                    mark_verified(st.session_state.pending_email)
                    st.success("✅ Email verified! Redirecting to login...")
                    time.sleep(1)
                    switch_page('login')
                else:
                    st.error(msg)
        with c2:
            if st.button("Resend OTP"):
                otp = generate_otp()
                store_otp(st.session_state.pending_email, otp, 'verification')
                send_otp_email(st.session_state.pending_email, otp, 'verification')
                st.success("New OTP sent!")
        st.markdown('</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════ FORGOT PASSWORD ══
def forgot_password_page():
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("<h1>🔑 Password Recovery</h1>", unsafe_allow_html=True)
        step = st.session_state.get('reset_step', 1)
        st.progress(step / 3)
        st.caption(f"Step {step} of 3")
        st.markdown('<div class="neon-card">', unsafe_allow_html=True)

        if step == 1:
            st.markdown("### Step 1 – Enter Email")
            email = st.text_input("Registered Email *")
            if st.button("Send OTP", type="primary"):
                user = get_user_by_email(email)
                if not user:
                    st.error("Email not found.")
                else:
                    otp = generate_otp()
                    store_otp(email, otp, 'reset')
                    send_otp_email(email, otp, 'reset')
                    st.session_state['reset_email'] = email
                    st.session_state['reset_user']  = user
                    st.session_state['reset_step']  = 2
                    st.success("OTP sent!")
                    time.sleep(0.5)
                    st.rerun()

        elif step == 2:
            st.markdown("### Step 2 – Verify Identity")
            user = st.session_state['reset_user']
            st.info(f"Account: **{st.session_state['reset_email']}**")
            otp_in  = st.text_input("Enter OTP *", max_chars=6)
            st.markdown(f"**Security Question:** {user[4]}")
            sec_ans = st.text_input("Security Answer *")
            c1, c2 = st.columns(2)
            with c1:
                if st.button("Verify", type="primary"):
                    otp_ok, otp_msg = verify_otp(st.session_state['reset_email'], otp_in, 'reset')
                    ans_ok = verify_password(sec_ans.lower(), user[5])
                    if not otp_ok:
                        st.error(f"OTP: {otp_msg}")
                    elif not ans_ok:
                        st.error("Incorrect security answer.")
                    else:
                        st.session_state['reset_step'] = 3
                        st.rerun()
            with c2:
                if st.button("Resend OTP"):
                    otp = generate_otp()
                    store_otp(st.session_state['reset_email'], otp, 'reset')
                    send_otp_email(st.session_state['reset_email'], otp, 'reset')
                    st.success("Resent!")

        elif step == 3:
            st.markdown("### Step 3 – Set New Password")
            st.warning("⚠️ You cannot reuse any of your previous passwords.")
            p1 = st.text_input("New Password *", type="password")
            p2 = st.text_input("Confirm Password *", type="password")
            if p1: show_strength(p1)
            if st.button("Update Password", type="primary"):
                if p1 != p2:
                    st.error("Passwords do not match.")
                elif not validate_password(p1):
                    st.error("Password must have letters and numbers and be 6+ chars.")
                elif is_password_reused(st.session_state['reset_email'], p1):
                    st.error("⚠️ Old password reuse is not permitted.")
                else:
                    s, _ = check_password_strength(p1)
                    if s == "Weak":
                        st.error("Password is too weak.")
                    else:
                        ok, _ = update_password(st.session_state['reset_email'], p1)
                        if ok:
                            st.balloons()
                            st.success("Password updated! Please login.")
                            for k in ['reset_email', 'reset_user', 'reset_step']:
                                st.session_state.pop(k, None)
                            time.sleep(1.5)
                            switch_page('login')

        st.markdown("---")
        if st.button("Cancel – Back to Login"):
            for k in ['reset_email', 'reset_user', 'reset_step']:
                st.session_state.pop(k, None)
            switch_page('login')
        st.markdown('</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════════ ADMIN LOGIN ══
def admin_login_page():
    col1, col2, col3 = st.columns([1, 1.5, 1])
    with col2:
        st.markdown("<h1>🛡️ Admin Portal</h1>", unsafe_allow_html=True)
        st.markdown('<div class="neon-card">', unsafe_allow_html=True)
        with st.form("admin_form"):
            email = st.text_input("Admin Email *")
            pw    = st.text_input("Admin Password *", type="password")
            if st.form_submit_button("Login as Admin", type="primary"):
                adm_email, adm_pw = get_admin_creds()
                if email == adm_email and pw == adm_pw:
                    token = generate_token(email, "Admin", 'admin')
                    st.session_state.update({
                        'logged_in': True, 'token': token,
                        'username': 'Admin', 'email': email, 'role': 'admin'
                    })
                    st.success("✅ Admin login successful!")
                    time.sleep(0.5)
                    st.rerun()
                else:
                    st.error("Invalid admin credentials.")
        if st.button("Back to User Login"): switch_page('login')
        st.markdown('</div>', unsafe_allow_html=True)


# ═══════════════════════════════════════════════════ AUTHENTICATED PAGES ══

def home_page():
    st.markdown(f"""
    <div style="background:#1f2937; border:1px solid #374151; border-left:4px solid #00ffcc;
                border-radius:12px; padding:30px; margin-bottom:24px;">
        <h2 style="margin:0; color:#00ffcc !important;">Welcome, {st.session_state.username}! ⚡</h2>
        <p style="color:#9ca3af; margin-top:8px;">{st.session_state.email}</p>
    </div>""", unsafe_allow_html=True)

    c1, c2, c3, c4 = st.columns(4)
    metrics = [
        ("Account", "Active"),
        ("Session", "JWT ✓"),
        ("2FA", "OTP ✓"),
        ("Role", st.session_state.role.capitalize()),
    ]
    for col, (lbl, val) in zip([c1, c2, c3, c4], metrics):
        col.metric(lbl, val)

    st.markdown("---")
    st.markdown("""
    <div style="background:#1f2937; border:1px solid #374151; border-radius:10px; padding:20px;">
    <h3 style="color:#00ffcc !important; margin-top:0;">🎯 Milestone 2 – Features Active</h3>
    <ul style="color:#d1d5db; line-height:2;">
        <li>✅ OTP Email Authentication (signup + every login)</li>
        <li>✅ Account Lockout after 3 failed attempts (5-min timeout)</li>
        <li>✅ Forgot Password with no reuse of last 5 passwords</li>
        <li>✅ Password Strength Meter</li>
        <li>✅ Admin Dashboard with user management</li>
        <li>✅ Readability Analyzer with Plotly gauges + PDF/TXT upload</li>
    </ul>
    </div>""", unsafe_allow_html=True)


# ─── READABILITY PAGE ────────────────────────────────────────────────────────
def readability_page():
    st.markdown("<h1>📖 Text Readability Analyzer</h1>", unsafe_allow_html=True)
    st.markdown('<p style="color:#9ca3af;">Analyze how readable and accessible any policy document is for the general public.</p>', unsafe_allow_html=True)

    tab1, tab2 = st.tabs(["✍️  Input Text", "📂  Upload File (TXT / PDF)"])
    text_input = ""

    with tab1:
        raw_text = st.text_area("Enter text to analyze (min 50 characters):", height=220,
                                 placeholder="Paste policy document, government notice, or legal text here...")
        if raw_text:
            text_input = raw_text

    with tab2:
        uploaded_file = st.file_uploader("Upload a .txt or .pdf file", type=["txt", "pdf"])
        if uploaded_file:
            try:
                if uploaded_file.type == "application/pdf":
                    reader = PyPDF2.PdfReader(uploaded_file)
                    extracted = ""
                    for page in reader.pages:
                        extracted += (page.extract_text() or "") + "\n"
                    text_input = extracted
                    st.success(f"✅ Loaded {len(reader.pages)} page(s) from PDF.")
                else:
                    text_input = uploaded_file.read().decode("utf-8")
                    st.success(f"✅ Loaded: {uploaded_file.name}")
            except Exception as e:
                st.error(f"Error reading file: {e}")

    st.markdown("")
    if st.button("⚡ Analyze Readability", type="primary"):
        if len(text_input.strip()) < 50:
            st.error("Text is too short (minimum 50 characters). Please enter more text or upload a valid file.")
        else:
            with st.spinner("Computing readability metrics..."):
                analyzer = ReadabilityAnalyzer(text_input)
                score    = analyzer.get_all_metrics()

            st.markdown("---")
            st.subheader("📊 Analysis Results")

            # ── Overall Level ──────────────────────────────────────────────
            avg_grade = (
                score['Flesch-Kincaid Grade'] +
                score['Gunning Fog'] +
                score['SMOG Index'] +
                score['Coleman-Liau']
            ) / 4

            if avg_grade <= 6:
                level, color = "Beginner (Elementary)", "#28a745"
            elif avg_grade <= 10:
                level, color = "Intermediate (Middle School)", "#17a2b8"
            elif avg_grade <= 14:
                level, color = "Advanced (High School / College)", "#ffc107"
            else:
                level, color = "Expert (Professional / Academic)", "#dc3545"

            st.markdown(f"""
            <div class="neon-card-accent" style="border-left: 5px solid {color}; text-align:center;">
                <h2 style="margin:0; color:{color} !important;">Overall Level: {level}</h2>
                <p style="margin:6px 0 0; color:#9ca3af; font-family:'Share Tech Mono',monospace;">
                Approximate Grade Level: <strong style="color:{color};">{int(avg_grade)}</strong>
                </p>
            </div>""", unsafe_allow_html=True)

            # ── Plotly Gauges ─────────────────────────────────────────────
            st.markdown("### 📈 Detailed Metrics")

            g1, g2, g3 = st.columns(3)
            with g1:
                st.plotly_chart(
                    create_gauge(score["Flesch Reading Ease"],
                                 "Flesch Reading Ease", 0, 100, "#00ffcc"),
                    use_container_width=True)
                with st.expander("ℹ️ About Flesch Reading Ease"):
                    st.caption("0–100 scale. Higher = easier to read. 60–70 is standard for general audience.")

            with g2:
                st.plotly_chart(
                    create_gauge(score["Flesch-Kincaid Grade"],
                                 "Flesch-Kincaid Grade", 0, 20, "#ff00ff"),
                    use_container_width=True)
                with st.expander("ℹ️ About Flesch-Kincaid Grade"):
                    st.caption("US Grade Level. Score of 8 means an 8th grader can comfortably understand the text.")

            with g3:
                st.plotly_chart(
                    create_gauge(score["SMOG Index"],
                                 "SMOG Index", 0, 20, "#ffff00"),
                    use_container_width=True)
                with st.expander("ℹ️ About SMOG Index"):
                    st.caption("Commonly used in medical and health writing. Based on polysyllabic word count.")

            g4, g5 = st.columns(2)
            with g4:
                st.plotly_chart(
                    create_gauge(score["Gunning Fog"],
                                 "Gunning Fog Index", 0, 20, "#00ccff"),
                    use_container_width=True)
                with st.expander("ℹ️ About Gunning Fog"):
                    st.caption("Based on average sentence length and percentage of complex words (3+ syllables).")

            with g5:
                st.plotly_chart(
                    create_gauge(score["Coleman-Liau"],
                                 "Coleman-Liau Index", 0, 20, "#ff9900"),
                    use_container_width=True)
                with st.expander("ℹ️ About Coleman-Liau"):
                    st.caption("Character-based formula – works well for automated text analysis pipelines.")

            # ── Text Statistics ───────────────────────────────────────────
            st.markdown("### 📝 Text Statistics")
            s1, s2, s3, s4, s5 = st.columns(5)
            s1.metric("Sentences",     analyzer.num_sentences)
            s2.metric("Words",          analyzer.num_words)
            s3.metric("Syllables",      analyzer.num_syllables)
            s4.metric("Complex Words",  analyzer.complex_words)
            s5.metric("Characters",     analyzer.char_count)

            # ── Improvement Suggestions ───────────────────────────────────
            st.markdown("### 💡 Improvement Suggestions")
            sugg = analyzer.get_suggestions()
            for s in sugg:
                st.markdown(
                    f'<div class="neon-card" style="border-left:3px solid #00ffcc; padding:12px 18px;">'
                    f'{s}</div>', unsafe_allow_html=True)


# ─── ADMIN PAGE ──────────────────────────────────────────────────────────────
def admin_page():
    import pandas as pd
    import sqlite3

    st.markdown("<h1>🛡️ Admin Panel</h1>", unsafe_allow_html=True)
    tab_ov, tab_users, tab_otp = st.tabs(["📊 Overview", "👥 Users", "🔐 OTP Logs"])

    conn = sqlite3.connect('policynav_m2.db')

    with tab_ov:
        users_df = pd.read_sql_query("SELECT * FROM users", conn)
        otp_df   = pd.read_sql_query("SELECT * FROM otp_store", conn)
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Users",    len(users_df))
        c2.metric("Verified",       int(users_df['is_verified'].sum()) if len(users_df) else 0)
        c3.metric("Locked",         int(users_df['locked_until'].notna().sum()) if len(users_df) else 0)
        c4.metric("OTPs Sent",      len(otp_df))

    with tab_users:
        st.subheader("All Registered Users")
        df = pd.read_sql_query(
            "SELECT id, username, email, failed_attempts, is_verified, locked_until, created_at FROM users", conn)
        if len(df):
            def status(r):
                if r['locked_until']: return "🔒 Locked"
                if r['is_verified']:  return "✅ Verified"
                return "⏳ Pending"
            df['Status'] = df.apply(status, axis=1)
            st.dataframe(df[['id','username','email','Status','failed_attempts','created_at']],
                         use_container_width=True)
        else:
            st.info("No users registered yet.")

        st.markdown("---")
        st.subheader("🔓 Unlock Account")
        locked_df = pd.read_sql_query("SELECT email FROM users WHERE locked_until IS NOT NULL", conn)
        if len(locked_df):
            sel = st.selectbox("Select locked account", locked_df['email'].tolist())
            if st.button("Unlock Selected Account", type="primary"):
                reset_failed_attempts(sel)
                st.success(f"✅ {sel} unlocked!")
                st.rerun()
        else:
            st.success("No locked accounts currently.")

        st.markdown("---")
        st.subheader("🗑️ Delete User")
        all_emails = pd.read_sql_query(
            "SELECT email FROM users WHERE email != ?",
            conn, params=(st.session_state.email,))
        if len(all_emails):
            del_sel = st.selectbox("Select user to delete", all_emails['email'].tolist(), key="del")
            if st.button("Delete User", type="primary"):
                delete_user(del_sel)
                st.warning(f"Deleted {del_sel}")
                st.rerun()

    with tab_otp:
        st.subheader("OTP Audit Log (Latest 50)")
        odf = pd.read_sql_query(
            "SELECT email, purpose, used, expires_at FROM otp_store ORDER BY id DESC LIMIT 50", conn)
        odf['Status'] = odf['used'].map({1: "✅ Used", 0: "⏳ Pending"})
        st.dataframe(odf[['email','purpose','Status','expires_at']], use_container_width=True)

    conn.close()


# ═══════════════════════════════════════════════════════════════ ROUTER ══
def main():
    if st.session_state.logged_in:
        # ── Authenticated sidebar ────────────────────────────────────────
        with st.sidebar:
            try:
                st.image(
                    "https://upload.wikimedia.org/wikipedia/commons/thumb/9/95/Infosys_logo.svg/1280px-Infosys_logo.svg.png",
                    width=140)
            except:
                st.markdown("### ⚡ PolicyNav")
            st.markdown(f"**👤 {st.session_state.username}**")
            st.markdown(f"<small style='color:#9ca3af'>{st.session_state.email}</small>",
                        unsafe_allow_html=True)
            st.markdown("---")

            opts  = ["Home", "Readability"]
            icons = ["house", "book"]
            if st.session_state.role == 'admin':
                opts.append("Admin"); icons.append("shield-lock")

            selected = option_menu(
                "PolicyNav",
                opts,
                icons=icons,
                menu_icon="cast",
                default_index=0,
                styles={
                    "container":           {"background-color": "#111827"},
                    "icon":                {"color": "#00ffcc"},
                    "nav-link":            {"color": "#9ca3af",
                                           "font-family": "Share Tech Mono, monospace"},
                    "nav-link-selected":   {"background-color": "#00ffcc",
                                           "color": "#0e1117", "font-weight": "700"},
                }
            )
            st.markdown("---")
            if st.button("🔓 Log Out"): logout()

        if selected == "Home":         home_page()
        elif selected == "Readability": readability_page()
        elif selected == "Admin":       admin_page()

    else:
        # ── Unauthenticated routing ─────────────────────────────────────
        routes = {
            'login':             login_page,
            'signup':            signup_page,
            'verify_signup_otp': verify_signup_otp_page,
            'verify_login_otp':  verify_login_otp_page,
            'forgot_password':   forgot_password_page,
            'admin_login':       admin_login_page,
        }
        routes.get(st.session_state.page, login_page)()


if __name__ == "__main__":
    main()


Writing app.py


In [8]:
from pyngrok import ngrok, conf
from google.colab import userdata
import subprocess, time, os

# Load secrets
ngrok_token = userdata.get('NGROK_AUTHTOKEN')
jwt_secret  = userdata.get('JWT_SECRET_KEY')
email_pass  = userdata.get('EMAIL_APP_PASSWORD')

os.environ['JWT_SECRET']         = jwt_secret  or 'policynav-secret-key'
os.environ['EMAIL_APP_PASSWORD'] = email_pass  or ''

# Configure and start ngrok
conf.get_default().auth_token = ngrok_token
ngrok.kill()
time.sleep(1)

# Start Streamlit
process = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.port', '8501',
     '--server.headless', 'true'],
    env=os.environ.copy()
)
time.sleep(5)

# Open tunnel
public_url = ngrok.connect(8501).public_url
print('=' * 65)
print('PolicyNav Milestone 2 – LIVE!')
print('=' * 65)
print(f'\nPublic URL: {public_url}')
print('\nFeatures Active:')
print('  [+] Neon dark theme – Plotly gauge charts')
print('  [+] PDF and TXT file upload in Readability Analyzer')
print('  [+] OTP Email Authentication (signup + login)')
print('  [+] Account lockout after 3 failed attempts (5 min)')
print('  [+] No password reuse + password strength meter')
print('  [+] Admin Dashboard – users, OTP logs, delete user')
print('\nAdmin Login: use ADMIN_EMAIL_ID / ADMIN_PASSWORD from secrets')
print('=' * 65)

# Keep alive
try:
    input('\nPress ENTER to stop the app...\n')
except:
    pass
finally:
    process.terminate()
    ngrok.kill()
    print('Application stopped.')


PolicyNav Milestone 2 – LIVE!

Public URL: https://irrevocable-darell-martyrly.ngrok-free.dev

Features Active:
  [+] Neon dark theme – Plotly gauge charts
  [+] PDF and TXT file upload in Readability Analyzer
  [+] OTP Email Authentication (signup + login)
  [+] Account lockout after 3 failed attempts (5 min)
  [+] No password reuse + password strength meter
  [+] Admin Dashboard – users, OTP logs, delete user

Admin Login: use ADMIN_EMAIL_ID / ADMIN_PASSWORD from secrets

Press ENTER to stop the app...
enter
Application stopped.
